# chat — agent loop

Drives `graphs/chat` in-kernel. `MemorySaver` lives in this process, so the
conversation persists between cell runs and dies with the kernel — which is
exactly what you want while tuning.

Prints every tool call. A pretty transcript hides the thing you are debugging.

Needs `ollama serve` running.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
from dotenv import load_dotenv

sys.path.insert(0, "../src")
load_dotenv()

from langchain_core.messages import AIMessage, HumanMessage, ToolMessage

from uatu.graphs.chat.graph import graph
from uatu.prompts import load

PROJECT_ID = "6aac0013f45c3aedeec3e258"
THREAD = "dev-1"

print("prompt version:", load("chat").version)

## Driver

`ask` runs one turn and returns the answer plus a trace of what the model
called. `say` prints both.

In [ ]:
def _cfg(thread=None):
    return {
        "configurable": {
            # Namespaced: a thread can never be read from another project.
            "thread_id": f"{PROJECT_ID}:{thread or THREAD}",
            "project_id": PROJECT_ID,
        },
        # Seatbelt. A 14B model that loops hits this and raises.
        "recursion_limit": 12,
    }


def ask(text, thread=None):
    """Run one turn. Returns (answer, trace) without printing."""
    cfg = _cfg(thread)
    seen = len(graph.get_state(cfg).values.get("messages", []))
    result = graph.invoke({"messages": [HumanMessage(content=text)]}, config=cfg)

    trace = []
    for m in result["messages"][seen:]:
        if isinstance(m, AIMessage) and m.tool_calls:
            trace += [f"TOOL {c['name']}({c['args']})" for c in m.tool_calls]
        elif isinstance(m, ToolMessage):
            trace.append("  -> " + m.content.replace("\n", " ")[:150] + "...")

    return result["messages"][-1].content, trace


def say(text, thread=None):
    answer, trace = ask(text, thread)
    if trace:
        print("\n".join(trace))
    print(f"\n{'-' * 60}\n{answer}")

## Three turns

The third question says "it" — it only works if the checkpointer fed turn 2
back in. That is the real test of persistence.

In [ ]:
say("What does this project do?")

In [ ]:
say("Which file handles payment confirmation?")

In [ ]:
say("Read it and tell me if amount is guarded.")

## Tuning the prompt

`prompts.loader.load` is decorated with `@cache`, so **editing `chat.prompty`
does nothing in a running kernel**. `autoreload` will not save you — it reloads
modules, not a populated `functools.cache`.

Run this after every prompt edit:

In [ ]:
load.cache_clear()
print("reloaded, version:", load("chat").version)

Structural edits to `graph.py` need a kernel restart, because `graph` is
compiled at import. Edits to the *node functions* are picked up by autoreload —
the compiled graph holds references to the function objects and autoreload
patches them in place.

## Fresh conversation

No reset needed. `MemorySaver` keys by `thread_id`, so a new thread name is a
clean slate — useful for running the same question against two prompt versions.

In [ ]:
say("What does this project do?", thread="dev-2")

## Optional: the chat-looking one

Run this once the loop behaves. The trace is still rendered, just small — when
the agent answers wrongly, the tool calls are the only evidence of why.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, Markdown

log = widgets.Output()
entry = widgets.Text(
    placeholder="Ask about the project...",
    layout=widgets.Layout(width="100%"),
    # Not optional: observe on "value" fires on every change, so with the
    # default True, typing "hi" would launch two agent runs mid-word.
    continuous_update=False,
)


def _on_submit(change):
    q = (change["new"] or "").strip()
    if not q:
        return
    # Clearing re-enters this callback synchronously with new="". The guard
    # above absorbs it, which is why it comes before the clear.
    entry.value = ""

    with log:
        display(Markdown(f"**You:** {q}"))
    answer, trace = ask(q, thread="ui-1")
    with log:
        if trace:
            display(Markdown("<sub>" + "<br>".join(trace) + "</sub>"))
        display(Markdown(f"**Uatu:** {answer}"))


# on_submit was deprecated in ipywidgets 8.
entry.observe(_on_submit, names="value")
display(entry, log)